In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from pathlib import Path


# Using _Trajectory Container Tools_ with Direct Instantiation

This notebook demonstrates how to create trajectory containers directly from data arrays using TCT's dataclasses.


## 1. Basic Trajectory Container Creation

Import the required modules and create basic trajectory containers from data arrays.


In [ ]:
from trajectory_container_tools.trj_dataclasses.base_trajectory_dataclass import (
    BaseTrajectoryDataclass,
    BaseReverseAxisTrajectoryDataclass
)
from trajectory_container_tools.trj_dataclasses.abstract_trajectory_dataclass import (
    AbstractTrajectoryDataclass,
    AbstractMultifeatureDataclass
)

# Generate sample trajectory data
timesteps = 100
time_array = np.linspace(0, 10, timesteps)
x_positions = np.sin(time_array) * 3
y_positions = np.cos(time_array) * 2
timestamps = np.arange(timesteps) * 0.1  # 10 Hz sampling

# Create basic trajectory container
basic_trajectory = BaseTrajectoryDataclass(
    x=x_positions,
    y=y_positions,
    timestamps=timestamps
)

print(f"Created trajectory with {basic_trajectory.trajectory_len} timesteps")
print(f"Available dimensions: {basic_trajectory.get_dimension_names()}")
print(f"Data shapes: x={basic_trajectory.x.shape}, y={basic_trajectory.y.shape}")


## 2. Accessing and Manipulating Trajectory Data

Demonstrate different ways to access and manipulate trajectory data.


In [ ]:
# Access individual dimensions
x_data = basic_trajectory.x
y_data = basic_trajectory.y
time_data = basic_trajectory.timestamps

print(f"X position range: [{x_data.min():.2f}, {x_data.max():.2f}]")
print(f"Y position range: [{y_data.min():.2f}, {y_data.max():.2f}]")
print(f"Time range: [{time_data.min():.2f}, {time_data.max():.2f}] seconds")

# Slice trajectory (get timesteps 20-40)
partial_trajectory = basic_trajectory[20:40]
print(f"\nPartial trajectory length: {partial_trajectory.trajectory_len}")

# Iterate through first 5 trajectory points
print("\nFirst 5 trajectory points:")
for i, point in enumerate(basic_trajectory):
    if i >= 5:
        break
    x_val, y_val = point
    print(f"Point {i}: x={x_val:.3f}, y={y_val:.3f}")


## 3. Visualizing Basic Trajectories

Create visualizations of the trajectory data.


In [ ]:
plt.figure(figsize=(12, 8))

# Plot trajectory in 2D space
plt.subplot(2, 2, 1)
plt.plot(basic_trajectory.x, basic_trajectory.y, 'b-', linewidth=2, alpha=0.7)
plt.scatter(basic_trajectory.x[0], basic_trajectory.y[0], color='green', s=100, label='Start', zorder=5)
plt.scatter(basic_trajectory.x[-1], basic_trajectory.y[-1], color='red', s=100, label='End', zorder=5)
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('2D Trajectory')
plt.legend()
plt.grid(True)
plt.axis('equal')

# Plot X position over time
plt.subplot(2, 2, 2)
plt.plot(basic_trajectory.timestamps, basic_trajectory.x, 'r-', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('X Position')
plt.title('X Position vs Time')
plt.grid(True)

# Plot Y position over time
plt.subplot(2, 2, 3)
plt.plot(basic_trajectory.timestamps, basic_trajectory.y, 'g-', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Y Position')
plt.title('Y Position vs Time')
plt.grid(True)

# Plot both positions
plt.subplot(2, 2, 4)
plt.plot(basic_trajectory.timestamps, basic_trajectory.x, 'r-', linewidth=2, label='X Position')
plt.plot(basic_trajectory.timestamps, basic_trajectory.y, 'g-', linewidth=2, label='Y Position')
plt.xlabel('Time (s)')
plt.ylabel('Position')
plt.title('Position Components')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


## 4. Batch Trajectory Processing

Create and process multiple trajectories simultaneously.


In [ ]:
# Create batch trajectories (multiple trajectories at once)
batch_size = 5
time_steps = 80

# Generate different circular trajectories
batch_x = np.zeros((batch_size, time_steps))
batch_y = np.zeros((batch_size, time_steps))
batch_timestamps = np.tile(np.arange(time_steps) * 0.1, (batch_size, 1))

for i in range(batch_size):
    # Each trajectory has different radius and frequency
    radius = 1 + i * 0.5  # Increasing radius
    frequency = 0.5 + i * 0.2  # Increasing frequency
    t = np.linspace(0, 2*np.pi, time_steps)
    
    batch_x[i, :] = radius * np.cos(frequency * t)
    batch_y[i, :] = radius * np.sin(frequency * t)

# Create batch trajectory container
batch_trajectory = BaseTrajectoryDataclass(
    x=batch_x,
    y=batch_y,
    timestamps=batch_timestamps
)

print(f"Created batch trajectory with shape: {batch_trajectory.x.shape}")
print(f"Number of trajectories: {batch_trajectory.x.shape[0]}")
print(f"Timesteps per trajectory: {batch_trajectory.x.shape[1]}")

# Visualize batch trajectories
plt.figure(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0, 1, batch_size))

for i in range(batch_size):
    plt.plot(batch_trajectory.x[i, :], batch_trajectory.y[i, :], 
             color=colors[i], linewidth=2, alpha=0.8, 
             label=f'Trajectory {i+1}')
    
    # Mark start and end points
    plt.scatter(batch_trajectory.x[i, 0], batch_trajectory.y[i, 0], 
                color=colors[i], s=100, marker='o', edgecolor='black')
    plt.scatter(batch_trajectory.x[i, -1], batch_trajectory.y[i, -1], 
                color=colors[i], s=100, marker='s', edgecolor='black')

plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Batch Trajectories (Circles: Start, Squares: End)')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()


## 5. Custom Trajectory Dataclasses

Create custom trajectory dataclasses with specialized functionality.


In [ ]:
@dataclass
class RobotNavigationTrajectory(BaseTrajectoryDataclass):
    """Custom trajectory dataclass for robot navigation with additional fields"""
    x: np.ndarray
    y: np.ndarray
    theta: np.ndarray          # Robot orientation
    velocity: np.ndarray       # Linear velocity
    angular_velocity: np.ndarray # Angular velocity
    
    @classmethod
    def get_dimension_names(cls):
        return ['x', 'y', 'theta', 'velocity', 'angular_velocity']
    
    def post_init_feature_callback(self, feature_name: str):
        """Custom processing after initialization"""
        if feature_name == 'theta':
            # Wrap angles to [-pi, pi]
            theta = getattr(self, feature_name)
            wrapped_theta = np.arctan2(np.sin(theta), np.cos(theta))
            setattr(self, feature_name, wrapped_theta)
            print(f"Wrapped {feature_name} to [-π, π] range")
        
        elif feature_name == 'velocity':
            # Ensure velocities are non-negative
            velocity = getattr(self, feature_name)
            if np.any(velocity < 0):
                print(f"Warning: Found {np.sum(velocity < 0)} negative velocities")
                # Optionally fix negative velocities
                # velocity = np.abs(velocity)
                # setattr(self, feature_name, velocity)
        
        super().post_init_feature_callback(feature_name)
    
    def compute_metrics(self):
        """Compute trajectory metrics"""
        # Path length
        dx = np.diff(self.x, axis=-1 if self.current_trj_axe == -1 else 0)
        dy = np.diff(self.y, axis=-1 if self.current_trj_axe == -1 else 0)
        path_length = np.sum(np.sqrt(dx**2 + dy**2), axis=-1 if self.current_trj_axe == -1 else 0)
        
        # Average velocity
        avg_velocity = np.mean(self.velocity, axis=-1 if self.current_trj_axe == -1 else 0)
        
        # Maximum angular velocity
        max_angular_velocity = np.max(np.abs(self.angular_velocity), axis=-1 if self.current_trj_axe == -1 else 0)
        
        return {
            'path_length': path_length,
            'average_velocity': avg_velocity, 
            'max_angular_velocity': max_angular_velocity
        }

# Generate robot navigation data
nav_timesteps = 200
nav_time = np.linspace(0, 20, nav_timesteps)

# Create figure-8 trajectory
nav_x = 3 * np.sin(nav_time * 0.5)
nav_y = 2 * np.sin(nav_time * 1.0)  # Double frequency for figure-8
nav_theta = np.arctan2(np.gradient(nav_y), np.gradient(nav_x))  # Direction of motion
nav_velocity = np.sqrt(np.gradient(nav_x)**2 + np.gradient(nav_y)**2) / (nav_time[1] - nav_time[0])
nav_angular_velocity = np.gradient(nav_theta) / (nav_time[1] - nav_time[0])

# Create custom trajectory
robot_trajectory = RobotNavigationTrajectory(
    x=nav_x,
    y=nav_y,
    theta=nav_theta + 2*np.pi,  # Add 2π to test angle wrapping
    velocity=nav_velocity,
    angular_velocity=nav_angular_velocity,
    timestamps=nav_time
)

print(f"Robot trajectory: {robot_trajectory.trajectory_len} timesteps")

# Compute and display metrics
metrics = robot_trajectory.compute_metrics()
print(f"\nTrajectory Metrics:")
print(f"  Path length: {metrics['path_length']:.2f}")
print(f"  Average velocity: {metrics['average_velocity']:.3f}")
print(f"  Max angular velocity: {metrics['max_angular_velocity']:.3f}")


## 6. Visualizing Robot Navigation Trajectory

Create comprehensive visualizations of the robot navigation data.


In [ ]:
plt.figure(figsize=(15, 10))

# 2D trajectory with orientation arrows
plt.subplot(2, 3, 1)
plt.plot(robot_trajectory.x, robot_trajectory.y, 'b-', linewidth=2, alpha=0.7)

# Add orientation arrows (every 20 points)
arrow_spacing = 20
for i in range(0, len(robot_trajectory.x), arrow_spacing):
    dx = 0.3 * np.cos(robot_trajectory.theta[i])
    dy = 0.3 * np.sin(robot_trajectory.theta[i])
    plt.arrow(robot_trajectory.x[i], robot_trajectory.y[i], dx, dy,
              head_width=0.1, head_length=0.1, fc='red', ec='red', alpha=0.7)

plt.scatter(robot_trajectory.x[0], robot_trajectory.y[0], color='green', s=150, label='Start', zorder=5)
plt.scatter(robot_trajectory.x[-1], robot_trajectory.y[-1], color='red', s=150, label='End', zorder=5)
plt.xlabel('X Position (m)')
plt.ylabel('Y Position (m)')
plt.title('Robot Trajectory with Orientation')
plt.legend()
plt.grid(True)
plt.axis('equal')

# Position components over time
plt.subplot(2, 3, 2)
plt.plot(robot_trajectory.timestamps, robot_trajectory.x, 'r-', linewidth=2, label='X')
plt.plot(robot_trajectory.timestamps, robot_trajectory.y, 'g-', linewidth=2, label='Y')
plt.xlabel('Time (s)')
plt.ylabel('Position (m)')
plt.title('Position vs Time')
plt.legend()
plt.grid(True)

# Orientation over time
plt.subplot(2, 3, 3)
plt.plot(robot_trajectory.timestamps, robot_trajectory.theta, 'purple', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Orientation (rad)')
plt.title('Robot Orientation')
plt.grid(True)
plt.axhline(y=np.pi, color='k', linestyle='--', alpha=0.5)
plt.axhline(y=-np.pi, color='k', linestyle='--', alpha=0.5)

# Linear velocity
plt.subplot(2, 3, 4)
plt.plot(robot_trajectory.timestamps, robot_trajectory.velocity, 'orange', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Velocity (m/s)')
plt.title('Linear Velocity')
plt.grid(True)

# Angular velocity
plt.subplot(2, 3, 5)
plt.plot(robot_trajectory.timestamps, robot_trajectory.angular_velocity, 'cyan', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Angular Velocity (rad/s)')
plt.title('Angular Velocity')
plt.grid(True)

# Velocity profile (speed and direction)
plt.subplot(2, 3, 6)
speed = robot_trajectory.velocity
direction_change = np.abs(robot_trajectory.angular_velocity)
plt.plot(robot_trajectory.timestamps, speed, 'blue', linewidth=2, label='Speed')
plt.plot(robot_trajectory.timestamps, direction_change, 'red', linewidth=2, alpha=0.7, label='|Angular Vel|')
plt.xlabel('Time (s)')
plt.ylabel('Magnitude')
plt.title('Speed and Direction Changes')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


## 7. Multi-Feature Trajectory Container

Create a container that holds multiple related trajectory features.


In [ ]:
@dataclass
class MultiSensorTrajectoryContainer(AbstractMultifeatureDataclass):
    """Container for multi-sensor trajectory data"""
    pose: BaseTrajectoryDataclass
    velocity: BaseTrajectoryDataclass
    imu: BaseTrajectoryDataclass
    
    def __post_init__(self):
        super().__post_init__()
        
    def summary(self):
        """Print summary of all trajectory features"""
        print("Multi-sensor trajectory container:")
        print(f"  - Pose: {self.pose.trajectory_len} timesteps")
        print(f"  - Velocity: {self.velocity.trajectory_len} timesteps")
        print(f"  - IMU: {self.imu.trajectory_len} timesteps")
        
        # Check time alignment
        pose_duration = self.pose.timestamps[-1] - self.pose.timestamps[0]
        velocity_duration = self.velocity.timestamps[-1] - self.velocity.timestamps[0]
        imu_duration = self.imu.timestamps[-1] - self.imu.timestamps[0]
        
        print(f"  - Duration alignment: pose={pose_duration:.1f}s, velocity={velocity_duration:.1f}s, imu={imu_duration:.1f}s")

# Create individual trajectory components
multi_timesteps = 150
multi_time = np.linspace(0, 15, multi_timesteps)

# Pose data (robot navigation)
pose_x = 2 * np.sin(multi_time * 0.3)
pose_y = 1.5 * np.cos(multi_time * 0.6)
pose_theta = np.arctan2(np.gradient(pose_y), np.gradient(pose_x))

pose_data = BaseTrajectoryDataclass(
    x=pose_x,
    y=pose_y,
    theta=pose_theta,
    timestamps=multi_time
)

# Velocity data
vel_x = np.gradient(pose_x) / (multi_time[1] - multi_time[0])
vel_y = np.gradient(pose_y) / (multi_time[1] - multi_time[0])
vel_angular = np.gradient(pose_theta) / (multi_time[1] - multi_time[0])

velocity_data = BaseTrajectoryDataclass(
    vx=vel_x,
    vy=vel_y,
    vtheta=vel_angular,
    timestamps=multi_time
)

# IMU data (accelerometer and gyroscope)
# Add some realistic noise and dynamics
accel_x = np.gradient(vel_x) / (multi_time[1] - multi_time[0]) + np.random.normal(0, 0.1, multi_timesteps)
accel_y = np.gradient(vel_y) / (multi_time[1] - multi_time[0]) + np.random.normal(0, 0.1, multi_timesteps)
gyro_z = vel_angular + np.random.normal(0, 0.05, multi_timesteps)

imu_data = BaseTrajectoryDataclass(
    accel_x=accel_x,
    accel_y=accel_y,
    gyro_z=gyro_z,
    timestamps=multi_time
)

# Combine into multi-feature container
multi_sensor_container = MultiSensorTrajectoryContainer(
    pose=pose_data,
    velocity=velocity_data,
    imu=imu_data
)

print("Created multi-sensor trajectory container:")
multi_sensor_container.summary()


## 8. Visualizing Multi-Sensor Data

Create comprehensive visualizations showing all sensor data together.


In [ ]:
plt.figure(figsize=(16, 12))

# Trajectory with velocity vectors
plt.subplot(3, 4, 1)
plt.plot(multi_sensor_container.pose.x, multi_sensor_container.pose.y, 'b-', linewidth=2, alpha=0.7)

# Add velocity vectors (every 15 points)
vector_spacing = 15
scale = 0.1
for i in range(0, len(multi_sensor_container.pose.x), vector_spacing):
    vx = multi_sensor_container.velocity.vx[i]
    vy = multi_sensor_container.velocity.vy[i]
    plt.arrow(multi_sensor_container.pose.x[i], multi_sensor_container.pose.y[i], 
              scale*vx, scale*vy,
              head_width=0.05, head_length=0.05, fc='red', ec='red', alpha=0.7)

plt.xlabel('X Position (m)')
plt.ylabel('Y Position (m)')
plt.title('Trajectory with Velocity Vectors')
plt.grid(True)
plt.axis('equal')

# Position components
plt.subplot(3, 4, 2)
plt.plot(multi_sensor_container.pose.timestamps, multi_sensor_container.pose.x, 'r-', linewidth=2, label='X')
plt.plot(multi_sensor_container.pose.timestamps, multi_sensor_container.pose.y, 'g-', linewidth=2, label='Y')
plt.xlabel('Time (s)')
plt.ylabel('Position (m)')
plt.title('Position Components')
plt.legend()
plt.grid(True)

# Orientation
plt.subplot(3, 4, 3)
plt.plot(multi_sensor_container.pose.timestamps, multi_sensor_container.pose.theta, 'purple', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Orientation (rad)')
plt.title('Robot Orientation')
plt.grid(True)

# Linear velocities
plt.subplot(3, 4, 4)
plt.plot(multi_sensor_container.velocity.timestamps, multi_sensor_container.velocity.vx, 'r-', linewidth=2, label='Vx')
plt.plot(multi_sensor_container.velocity.timestamps, multi_sensor_container.velocity.vy, 'g-', linewidth=2, label='Vy')
plt.xlabel('Time (s)')
plt.ylabel('Velocity (m/s)')
plt.title('Linear Velocities')
plt.legend()
plt.grid(True)

# Angular velocity
plt.subplot(3, 4, 5)
plt.plot(multi_sensor_container.velocity.timestamps, multi_sensor_container.velocity.vtheta, 'orange', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Angular Velocity (rad/s)')
plt.title('Angular Velocity')
plt.grid(True)

# Speed magnitude
plt.subplot(3, 4, 6)
speed = np.sqrt(multi_sensor_container.velocity.vx**2 + multi_sensor_container.velocity.vy**2)
plt.plot(multi_sensor_container.velocity.timestamps, speed, 'blue', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Speed (m/s)')
plt.title('Speed Magnitude')
plt.grid(True)

# IMU accelerations
plt.subplot(3, 4, 7)
plt.plot(multi_sensor_container.imu.timestamps, multi_sensor_container.imu.accel_x, 'r-', linewidth=2, label='Accel X')
plt.plot(multi_sensor_container.imu.timestamps, multi_sensor_container.imu.accel_y, 'g-', linewidth=2, label='Accel Y')
plt.xlabel('Time (s)')
plt.ylabel('Acceleration (m/s²)')
plt.title('IMU Accelerations')
plt.legend()
plt.grid(True)

# IMU gyroscope
plt.subplot(3, 4, 8)
plt.plot(multi_sensor_container.imu.timestamps, multi_sensor_container.imu.gyro_z, 'cyan', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Angular Rate (rad/s)')
plt.title('IMU Gyroscope Z')
plt.grid(True)

# Acceleration magnitude
plt.subplot(3, 4, 9)
accel_mag = np.sqrt(multi_sensor_container.imu.accel_x**2 + multi_sensor_container.imu.accel_y**2)
plt.plot(multi_sensor_container.imu.timestamps, accel_mag, 'magenta', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('|Acceleration| (m/s²)')
plt.title('Acceleration Magnitude')
plt.grid(True)

# Correlation plot: Speed vs Acceleration
plt.subplot(3, 4, 10)
# Resample to same time points for comparison
common_time = multi_sensor_container.pose.timestamps
speed_interp = np.interp(common_time, multi_sensor_container.velocity.timestamps, speed)
accel_interp = np.interp(common_time, multi_sensor_container.imu.timestamps, accel_mag)

plt.scatter(speed_interp, accel_interp, alpha=0.6, s=20)
plt.xlabel('Speed (m/s)')
plt.ylabel('Acceleration Mag (m/s²)')
plt.title('Speed vs Acceleration')
plt.grid(True)

# Data quality check: time alignment
plt.subplot(3, 4, 11)
plt.plot(multi_sensor_container.pose.timestamps, np.ones_like(multi_sensor_container.pose.timestamps), 
         'ro', markersize=2, alpha=0.7, label='Pose')
plt.plot(multi_sensor_container.velocity.timestamps, np.ones_like(multi_sensor_container.velocity.timestamps) + 0.1, 
         'go', markersize=2, alpha=0.7, label='Velocity')
plt.plot(multi_sensor_container.imu.timestamps, np.ones_like(multi_sensor_container.imu.timestamps) + 0.2, 
         'bo', markersize=2, alpha=0.7, label='IMU')
plt.xlabel('Time (s)')
plt.ylabel('Sensor')
plt.title('Timestamp Alignment')
plt.legend()
plt.grid(True)

# Summary statistics
plt.subplot(3, 4, 12)
plt.text(0.1, 0.9, f"Trajectory Duration: {multi_time[-1]:.1f} s", transform=plt.gca().transAxes)
plt.text(0.1, 0.8, f"Sampling Rate: {1/(multi_time[1]-multi_time[0]):.1f} Hz", transform=plt.gca().transAxes)
plt.text(0.1, 0.7, f"Path Length: {np.sum(np.sqrt(np.diff(pose_x)**2 + np.diff(pose_y)**2)):.2f} m", transform=plt.gca().transAxes)
plt.text(0.1, 0.6, f"Max Speed: {np.max(speed):.2f} m/s", transform=plt.gca().transAxes)
plt.text(0.1, 0.5, f"Max Acceleration: {np.max(accel_mag):.2f} m/s²", transform=plt.gca().transAxes)
plt.text(0.1, 0.4, f"Pose Points: {len(multi_sensor_container.pose.timestamps)}", transform=plt.gca().transAxes)
plt.text(0.1, 0.3, f"Velocity Points: {len(multi_sensor_container.velocity.timestamps)}", transform=plt.gca().transAxes)
plt.text(0.1, 0.2, f"IMU Points: {len(multi_sensor_container.imu.timestamps)}", transform=plt.gca().transAxes)
plt.title('Summary Statistics')
plt.axis('off')

plt.tight_layout()
plt.show()


## 9. Data Validation and Error Handling

Demonstrate validation and error handling capabilities.


In [ ]:
@dataclass
class ValidatedTrajectoryDataclass(BaseTrajectoryDataclass):
    """Trajectory dataclass with built-in validation"""
    
    def post_init_feature_callback(self, feature_name: str):
        """Custom validation for each feature"""
        feature = getattr(self, feature_name)
        
        if isinstance(feature, np.ndarray):
            # Check for NaN values
            if np.any(np.isnan(feature)):
                raise ValueError(f"NaN values detected in {feature_name}")
            
            # Check for infinite values
            if np.any(np.isinf(feature)):
                raise ValueError(f"Infinite values detected in {feature_name}")
            
            # Feature-specific validation
            if feature_name == 'timestamps':
                # Check monotonicity
                if not np.all(np.diff(feature) >= 0):
                    raise ValueError("Timestamps are not monotonic")
                
                # Check for reasonable time values
                if np.any(feature < 0):
                    raise ValueError("Negative timestamps detected")
            
            elif feature_name in ['x', 'y'] and (np.any(np.abs(feature) > 1000)):
                raise ValueError(f"{feature_name} positions out of reasonable range (>1000m)")
            
            elif feature_name == 'velocity' and (np.any(feature < 0)):
                print(f"Warning: Negative velocities detected in {feature_name}")
        
        super().post_init_feature_callback(feature_name)

# Test with valid data
print("Testing with valid data:")
try:
    valid_trajectory = ValidatedTrajectoryDataclass(
        x=np.array([0, 1, 2, 3, 4]),
        y=np.array([0, 1, 0, -1, 0]),
        velocity=np.array([0, 1, 1, 1, 0.5]),
        timestamps=np.array([0, 1, 2, 3, 4])
    )
    print("✓ Validation passed for valid data")
    
except ValueError as e:
    print(f"✗ Validation failed: {e}")

# Test with invalid data (NaN values)
print("\nTesting with NaN values:")
try:
    invalid_trajectory_nan = ValidatedTrajectoryDataclass(
        x=np.array([0, 1, np.nan, 3, 4]),
        y=np.array([0, 1, 2, 3, 4]),
        timestamps=np.array([0, 1, 2, 3, 4])
    )
    print("✓ Validation passed (unexpected)")
    
except ValueError as e:
    print(f"✓ Validation correctly caught error: {e}")

# Test with invalid data (non-monotonic timestamps)
print("\nTesting with non-monotonic timestamps:")
try:
    invalid_trajectory_time = ValidatedTrajectoryDataclass(
        x=np.array([0, 1, 2, 3, 4]),
        y=np.array([0, 1, 2, 3, 4]),
        timestamps=np.array([0, 1, 3, 2, 4])  # 3, 2 is not monotonic
    )
    print("✓ Validation passed (unexpected)")
    
except ValueError as e:
    print(f"✓ Validation correctly caught error: {e}")


## 10. Performance Considerations and Best Practices

Demonstrate efficient ways to work with trajectory data.


In [ ]:
# Memory-efficient trajectory creation for large datasets
def create_memory_efficient_trajectory(length=100000):
    """Create large trajectory using memory-efficient techniques"""
    
    # Use appropriate data types
    timestamps = np.arange(length, dtype=np.float32) * 0.01  # 100 Hz
    
    # Generate data in chunks to avoid memory spikes
    chunk_size = 10000
    x_data = np.zeros(length, dtype=np.float32)
    y_data = np.zeros(length, dtype=np.float32)
    
    for i in range(0, length, chunk_size):
        end_idx = min(i + chunk_size, length)
        chunk_t = timestamps[i:end_idx]
        
        # Generate trajectory chunk
        x_data[i:end_idx] = np.sin(chunk_t * 0.1) * 5
        y_data[i:end_idx] = np.cos(chunk_t * 0.1) * 3
    
    trajectory = BaseTrajectoryDataclass(
        x=x_data,
        y=y_data,
        timestamps=timestamps
    )
    
    return trajectory

# Create and test large trajectory
print("Creating large trajectory (100k points)...")
large_traj = create_memory_efficient_trajectory(100000)
print(f"✓ Created trajectory with {large_traj.trajectory_len} points")
print(f"Memory usage: ~{(large_traj.x.nbytes + large_traj.y.nbytes + large_traj.timestamps.nbytes) / 1024**2:.1f} MB")

# Demonstrate efficient slicing
print("\nDemonstrating efficient slicing:")
slice_start = 10000
slice_end = 20000
trajectory_slice = large_traj[slice_start:slice_end]
print(f"Sliced trajectory: {trajectory_slice.trajectory_len} points")

# Show trajectory statistics
print(f"\nTrajectory Statistics:")
print(f"X range: [{large_traj.x.min():.2f}, {large_traj.x.max():.2f}]")
print(f"Y range: [{large_traj.y.min():.2f}, {large_traj.y.max():.2f}]")
print(f"Duration: {large_traj.timestamps[-1] - large_traj.timestamps[0]:.1f} seconds")
print(f"Sample rate: {1/(large_traj.timestamps[1] - large_traj.timestamps[0]):.1f} Hz")


## Summary

This notebook demonstrated the key features of Trajectory Container Tools for direct instantiation:

1. **Basic Usage**: Creating simple trajectory containers from data arrays
2. **Data Access**: Various ways to access and manipulate trajectory data
3. **Visualization**: Comprehensive plotting techniques for trajectory analysis
4. **Batch Processing**: Handling multiple trajectories simultaneously
5. **Custom Dataclasses**: Creating specialized trajectory containers with validation
6. **Multi-Feature Containers**: Combining multiple related trajectory features
7. **Validation**: Built-in and custom validation techniques
8. **Performance**: Memory-efficient approaches for large datasets

### Key Takeaways:
- TCT provides flexible and type-safe trajectory containers
- Custom dataclasses enable domain-specific functionality
- Built-in validation ensures data integrity
- Efficient memory management supports large datasets
- Rich visualization capabilities aid in data analysis

### Next Steps:
- Explore the DataFrame usage examples for pandas integration
- Try the ROS bag examples for robotics data processing
- Check out the full documentation for advanced features


In [ ]:
print("🎉 Trajectory Container Tools Direct Instantiation Tutorial Complete!")
print("Visit the documentation for more advanced examples and best practices.")
